# Avro Basics — 03: Schema Definition


Avro schemas are defined in JSON. Understanding Avro types is essential
for building correct schemas for your data.

Topics: primitive types, records, arrays, maps, unions, enums, fixed,
converting between Avro JSON schema and Spark StructType.


In [ ]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
# Re-suppress noisy loggers after setLogLevel resets them (Spark 4.x / log4j2)
_jvm = spark.sparkContext._jvm
_ctx = _jvm.org.apache.logging.log4j.LogManager.getContext(False)
_cfg = _ctx.getConfiguration()
for _logger_name in [
    'org.apache.iceberg.hadoop.HadoopTableOperations',
    'org.apache.hadoop.util.NativeCodeLoader',
]:
    _lc = _jvm.org.apache.logging.log4j.core.config.LoggerConfig(_logger_name,
              _jvm.org.apache.logging.log4j.Level.ERROR, False)
    _cfg.addLogger(_logger_name, _lc)
_ctx.updateLoggers()
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

## Step 1 — Avro Primitive Types



In [ ]:

print("""
Avro primitive types:
  null    → NullType
  boolean → BooleanType
  int     → IntegerType
  long    → LongType
  float   → FloatType
  double  → DoubleType
  bytes   → BinaryType
  string  → StringType

Logical types (annotations on primitives):
  {"type":"long","logicalType":"timestamp-micros"} → TimestampType
  {"type":"int", "logicalType":"date"}             → DateType
  {"type":"long","logicalType":"time-micros"}      → no direct Spark mapping
""")


## Step 2 — Record Type (Nested Struct)



In [ ]:
import json as pyjson
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType

RECORD_SCHEMA = pyjson.dumps({
    "type": "record",
    "name": "Order",
    "namespace": "com.example",
    "fields": [
        {"name": "order_id",  "type": "long"},
        {"name": "customer",  "type": {
            "type": "record", "name": "Customer",
            "fields": [
                {"name": "id",      "type": "long"},
                {"name": "name",    "type": "string"},
                {"name": "country", "type": "string"},
            ]
        }},
        {"name": "amount",    "type": "double"},
        {"name": "status",    "type": "string"},
    ]
})

# Must use explicit StructType — createDataFrame with tuples creates _1,_2,_3
# which doesn't match the Avro schema field names (id, name, country)
nested_schema = StructType([
    StructField("order_id", LongType()),
    StructField("customer", StructType([
        StructField("id",      LongType()),
        StructField("name",    StringType()),
        StructField("country", StringType()),
    ])),
    StructField("amount", DoubleType()),
    StructField("status", StringType()),
])

from pyspark.sql import Row
nested_data = spark.createDataFrame([
    Row(order_id=1, customer=Row(id=101, name="Alice", country="US"), amount=999.99, status="confirmed"),
    Row(order_id=2, customer=Row(id=102, name="Bob",   country="UK"), amount=499.99, status="shipped"),
    Row(order_id=3, customer=Row(id=103, name="Carol", country="DE"), amount=299.99, status="pending"),
], schema=nested_schema)

OUT = f"{DATA_DIR}/nested_record"
nested_data.write.format("avro").option("avroSchema", RECORD_SCHEMA).mode("overwrite").save(OUT)
print("Written nested Avro record:")
df_back = spark.read.format("avro").load(OUT)
df_back.show(truncate=False)
df_back.printSchema()

## Step 3 — Array and Map Types



In [ ]:

arr_schema = pyjson.dumps({
    "type": "record", "name": "Product",
    "fields": [
        {"name": "id",     "type": "long"},
        {"name": "tags",   "type": {"type": "array",  "items": "string"}},
        {"name": "prices", "type": {"type": "map",    "values": "double"}},
    ]
})

arr_data = spark.createDataFrame([
    (1, ["tech","premium"], {"USD": 999.99, "EUR": 919.99}),
    (2, ["budget"],         {"USD": 29.99,  "EUR": 27.99}),
], ["id","tags","prices"])

OUT_ARR = f"{DATA_DIR}/arrays_maps"
arr_data.write.format("avro").option("avroSchema", arr_schema).mode("overwrite").save(OUT_ARR)
print("Avro with arrays and maps:")
spark.read.format("avro").load(OUT_ARR).show(truncate=False)


## Step 4 — Union (Nullable Fields)

Union with null is how Avro represents nullable fields.

In [ ]:
union_schema = pyjson.dumps({
    "type": "record", "name": "Event",
    "fields": [
        {"name": "id",      "type": "long"},
        {"name": "value",   "type": ["null","double"],  "default": None},
        {"name": "label",   "type": ["null","string"],  "default": None},
        {"name": "tag",     "type": ["string","null"],  "default": "unknown"},
    ]
})

# Must use explicit schema — tuples with None need nullable StructField
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType
from pyspark.sql import Row

union_schema_spark = StructType([
    StructField("id",    LongType(),   nullable=False),
    StructField("value", DoubleType(), nullable=True),
    StructField("label", StringType(), nullable=True),
    StructField("tag",   StringType(), nullable=True),
])

union_data = spark.createDataFrame([
    Row(id=1, value=99.99, label="gold",   tag="vip"),
    Row(id=2, value=None,  label=None,     tag="standard"),
    Row(id=3, value=49.99, label="silver", tag=None),
], schema=union_schema_spark)

OUT_U = f"{DATA_DIR}/unions"
union_data.write.format("avro").option("avroSchema", union_schema).mode("overwrite").save(OUT_U)
df_u = spark.read.format("avro").load(OUT_U)
print("Avro union (nullable fields):")
df_u.show()
df_u.printSchema()
print('["null","string"] in Avro -> StringType nullable=true in Spark')

## Step 5 — StructType to Avro Schema Conversion



In [ ]:

from pyspark.sql.avro.functions import to_avro, from_avro

# Spark can serialize a column as Avro binary using to_avro
sample = spark.createDataFrame([(1,"Alice",99.99),(2,"Bob",49.99)], ["id","name","amount"])
avro_binary = sample.select(to_avro(F.struct("*")).alias("avro_bytes"))
print("Serialized as Avro binary column:")
avro_binary.show(truncate=False)
print("This is the Kafka Avro message format (without schema registry header).")


## Summary



In [ ]:

print("""
Avro type → Spark type mapping:
  null          → NullType
  boolean       → BooleanType
  int           → IntegerType
  long          → LongType
  float         → FloatType
  double        → DoubleType
  string        → StringType
  bytes         → BinaryType
  array[T]      → ArrayType(T)
  map[string,T] → MapType(StringType, T)
  record        → StructType
  ["null","T"]  → T (nullable)  ← most common nullable pattern
  enum          → StringType
  fixed         → BinaryType

  logicalType:date            → DateType
  logicalType:timestamp-micros → TimestampType
""")
